
# 🥇 Stage 4: Gold Layer (Star Schema & Struct Optimization)

Reads the Silver Delta table, loads dimension tables from BigQuery, performs a salted join on the calendar dimension to prevent data skew, organizes fields into optimized nested STRUCTs, and overwrites the target Gold BigQuery table.


In [ ]:
# ============================================================
# GOLD LAYER — Stage 4: Star Schema & BigQuery Write
# ============================================================
# Import PySpark functions for salted joins, broadcast hints,
# nested STRUCT construction, and derived KPI calculations.

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    broadcast, col, lit, current_timestamp, rand, floor,
    regexp_replace, explode, array, struct, coalesce
)


In [ ]:
# ------------------------------------------------------------
# PARAMETERS: Databricks widgets for parameterized execution.
#   env            : environment tag (dv/qa/pd)
#   pipeline_bucket: GCS bucket containing Silver Delta table
#                    and used as temp staging for BQ write
#   project_id     : GCP project ID for BigQuery
#   dataset        : BigQuery dataset name (default: mobile_brands)
#   run_id         : unique run identifier for traceability
# ------------------------------------------------------------

dbutils.widgets.text("env", "dv")
dbutils.widgets.text("pipeline_bucket", "")
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("dataset", "mobile_brands")
dbutils.widgets.text("run_id", "")

env = dbutils.widgets.get("env")
pipeline_bucket = dbutils.widgets.get("pipeline_bucket")
project_id = dbutils.widgets.get("project_id")
dataset = dbutils.widgets.get("dataset")
run_id = dbutils.widgets.get("run_id")

if not pipeline_bucket or not project_id:
    raise ValueError("Parameters 'pipeline_bucket' and 'project_id' are required.")


%run ./Credentials


In [ ]:
# ------------------------------------------------------------
# GCS & BIGQUERY AUTHENTICATION
# Injects service account credentials for both GCS (Silver read)
# and BigQuery (dimension reads + Gold write) into the Spark
# session. parentProject and credentials are required by the
# Spark BigQuery connector for indirect write mode.
# Enable adaptive query and increase shuffle partitions for joins.
# ------------------------------------------------------------

try:
    if 'gcp_key_json' in locals() and gcp_key_json.strip():
        spark.conf.set("google.cloud.auth.service.account.enable", "true")
        spark.conf.set("google.cloud.auth.service.account.json.keyfile.data", gcp_key_json)
        
        # Configure BigQuery credentials specifically for the connector
        spark.conf.set("parentProject", project_id)
        spark.conf.set("credentials", gcp_key_json)
        print("GCS and BigQuery Authentication configured using direct JSON key.")
    else:
        print("gcp_key_json variable is empty or not defined. Attempting default environment credentials.")
except Exception as e:
    print(f"Fallback to GCP Environment Default Auth / Error: {e}")

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "100")


In [ ]:
# ------------------------------------------------------------
# BIGQUERY READER HELPER: read_bq_table(table_name)
# Reads a BigQuery table via the Spark BigQuery connector.
# Used to load all four dimension tables (dim_date, dim_market,
# product_v2, customer) into Spark DataFrames for joining.
# ------------------------------------------------------------

SALT_BUCKETS = 10

def read_bq_table(table_name):
    # Reads BigQuery dimension tables using the spark bigquery connector
    return spark.read \
        .format("bigquery") \
        .option("table", f"{project_id}.{dataset}.{table_name}") \
        .load()


In [ ]:
# ------------------------------------------------------------
# DATA LOAD: Silver Fact + BigQuery Dimensions
# Reads the Silver Delta table (fact data) from GCS.
# Loads 4 BigQuery dimension tables:
#   dim_date   : calendar hierarchy (year/month/week/quarter)
#   dim_market : country/market metadata and hub currency
#   product_v2 : device specs (display, processor, camera, RAM)
#   customer   : channel hierarchy (distributor/retailer/store)
# product_v2 and customer are cached — reused across multiple joins.
# ------------------------------------------------------------

silver_path = f"gs://{pipeline_bucket}/delta/silver/"
print(f"Reading Silver Delta table from: {silver_path}")
fact_df = spark.read.format("delta").load(silver_path)

# Load BigQuery Dimensions
print(f"Loading dimensions from BigQuery dataset: {project_id}.{dataset}")
dim_calender = read_bq_table("dim_date")
dim_market = read_bq_table("dim_market")
dim_product = read_bq_table("product_v2").cache()
dim_customer = read_bq_table("customer").cache()


In [ ]:
# ------------------------------------------------------------
# SALTED JOIN PREPARATION (Anti-Data-Skew Pattern)
# Problem: dim_date is joined on date_reported which causes
#   data skew — all records for one date hit the same executor.
# Solution (Salt Buckets = 10):
#   Facts  → add a random salt value (0–9) per row.
#   Calendar → explode into 10 copies, one per salt bucket.
# The join key becomes (date + salt), distributing load evenly.
# dim_market, product, customer are aliased for broadcast joins.
# ------------------------------------------------------------

f_salted = fact_df \
    .withColumn("join_date", regexp_replace(col("date_reported").cast("string"), "-", "")) \
    .withColumn("salt", floor(rand() * SALT_BUCKETS)) \
    .alias("f")

# ── Prepare Calendar (explode salts to match salted facts)
c_salted = dim_calender \
    .withColumn("join_date", col("date_code").cast("string")) \
    .withColumn("salt_array", array([lit(i) for i in range(SALT_BUCKETS)])) \
    .select("*", explode(col("salt_array")).alias("salt")) \
    .alias("c")

m = dim_market.alias("m")
p = dim_product.alias("p")
c_cust = dim_customer.alias("c_cust")


In [ ]:
# ------------------------------------------------------------
# DIMENSION JOINS (Star Schema Assembly)
# Chains four left joins from the Silver fact table:
#   1. broadcast(dim_market)  on Market_code = market_code3
#   2. dim_calender (salted)  on join_date + salt (anti-skew)
#   3. broadcast(product_v2) on Brand + EAN_code
#   4. broadcast(customer)   on Store_code
# Left joins preserve all Silver rows even if a dim match is missing.
# ------------------------------------------------------------

joined_df = f_salted \
    .join(broadcast(m), col("f.Market_code") == col("m.market_code3"), "left") \
    .join(c_salted, (col("f.join_date") == col("c.join_date")) & (col("f.salt") == col("c.salt")), "left") \
    .join(broadcast(p), (col("f.Brand") == col("p.brand")) & (col("f.EAN_code") == col("p.ean_code")), "left") \
    .join(broadcast(c_cust), col("f.Store_code") == col("c_cust.store_code"), "left")


In [ ]:
# ------------------------------------------------------------
# GOLD SCHEMA: Nested STRUCT Construction
# Selects columns into nested BigQuery STRUCT types for optimal
# query performance (column pruning reduces BQ query cost).
# Derived KPIs computed inline:
#   sale_value  = Sale_units × Price
#   stock_value = Stock_units × Price
# STRUCTs: GEOGRAPHY, TRANSACTION_DATE, PRODUCT, CUSTOMER,
#          CURRENCY, FACT_VALUE, METADATA_TECHNICAL
# Repartitioned to 4 files to control BigQuery load job size.
# ------------------------------------------------------------

gold_df = joined_df.select(
    coalesce(col("f.Brand"), col("p.brand")).alias("tech_brand_name"),
    lit(None).cast("string").alias("brand_segment"),
    col("m.market_name").alias("tech_orga_country_name"),

    struct(
        col("m.market_name").alias("country_name"),
        col("m.market_code2").alias("market_code2"),
        col("m.market_code3").alias("market_code3"),
        lit(None).cast("string").alias("zone_code"),
        lit(None).cast("string").alias("zone_name"),
    ).alias("GEOGRAPHY"),

    struct(
        col("f.date_reported").alias("period_date"),
        col("c.year").alias("year"),
        col("c.month").alias("month"),
        col("c.week_num").alias("week_num"),
        col("c.month_year").alias("month_year"),
        col("c.quarter").alias("quarter"),
        col("c.quarter_year").alias("quarter_year"),
    ).alias("TRANSACTION_DATE"),

    struct(
        col("f.EAN_code").alias("product_code"),
        coalesce(col("f.Brand"), col("p.brand")).alias("brand"),
        col("f.Model").alias("model"),
        col("p.display_specification").alias("display"),
        col("p.processor_chipset").alias("processor"),
        col("p.front_camera").alias("rear_camera"),
        col("p.back_camera").alias("back_camera"),
        col("p.ram_gb").alias("ram"),
        col("p.rom_options").alias("rom"),
        col("p.refresh_rate_hz").alias("refresh_rate"),
        col("p.launch_year").alias("year_of_launch"),
    ).alias("PRODUCT"),

    struct(
        col("f.Distributor_code").alias("distributor_code"),
        col("c_cust.distributor_name").alias("distributor_name"),
        col("f.Retailer_code").alias("retailer_code"),
        col("c_cust.retailer_name").alias("retailer_name"),
        col("f.Store_code").alias("store_code"),
        col("c_cust.store_name").alias("store_name"),
        col("c_cust.channel_mode").alias("channel_mode"),
    ).alias("CUSTOMER"),

    struct(
        col("f.Currency").alias("currency"),
        col("m.hub_currency").alias("hub_currency"),
        lit(None).cast("string").alias("conversion_rate"),
    ).alias("CURRENCY"),

    struct(
        col("f.Price").alias("asp_local"),
        col("f.Sale_units").alias("units_sold"),
        (col("f.Sale_units") * col("f.Price")).alias("sale_value"),
        lit(None).cast("string").alias("total_sale_price_eur_value"),
        col("f.Stock_units").alias("stock_units"),
        (col("f.Stock_units") * col("f.Price")).alias("stock_value"),
        lit(None).cast("string").alias("total_stock_price_eur_value"),
    ).alias("FACT_VALUE"),

    struct(
        col("f.file_name").alias("file_name"),
        col("f.load_timestamp").alias("file_creation_date"),
        current_timestamp().alias("load_timestamp"),
    ).alias("METADATA_TECHNICAL"),
)

# Repartition to optimize write size
gold_df_partitioned = gold_df.repartition(4)


In [ ]:
# ------------------------------------------------------------
# WRITE TO BIGQUERY GOLD TABLE
# Writes the Gold DataFrame to BigQuery using indirect write mode:
#   Spark → GCS temp bucket (Parquet staging files)
#   GCS   → BigQuery load job (atomic table overwrite)
# This bypasses streaming insert quotas and is the recommended
# approach for large batch loads via the Spark BQ connector.
# Target: <project_id>.mobile_brands.gold_brand_daily_v1
# ------------------------------------------------------------

gold_bq_table = f"{project_id}.{dataset}.gold_brand_daily_v1"
print(f"Writing to BigQuery Table: {gold_bq_table}")

row_count = gold_df_partitioned.count()
print(f"Row count to write: {row_count}")

# Write using the spark bigquery connector in indirect mode
(
    gold_df_partitioned.write
    .format("bigquery")
    .option("table", gold_bq_table)
    .option("writeMethod", "indirect")
    .option("temporaryGcsBucket", pipeline_bucket)
    .mode("overwrite")
    .save()
)

dbutils.notebook.exit(f"Gold table written successfully. Row count: {row_count}")
